# Level 2, Notebook 01: Prepare Documents for RAG

**Problems in this notebook**

- Problem 11: Chunk Documents with Overlap
- Problem 19: Partial Document Ingestion
- Problem 20: PII Redaction Validation

**How to use this notebook**

1. Read the problem and say your plan aloud before you write any code.
2. Use the AI prompt given (or your own narrow one) to fill in the starter. Read the AI's code before you run it.
3. Run the checks. Open the reference solution only after you have tried.

The ideas behind these problems are in `01_prepare_documents_for_rag_THEORY.md`, next to this notebook.

In [ ]:
def run_checks(fn, cases):
    """cases: list of (args_tuple, expected). Prints a short report, never raises."""
    passed = 0
    for i, (args, expected) in enumerate(cases, 1):
        try:
            got = fn(*args)
        except NotImplementedError:
            print("Not implemented yet. Fill in the starter first.")
            return False
        except Exception as e:
            print(f"Case {i}: raised {type(e).__name__}: {e}")
            continue
        if got == expected:
            passed += 1
        else:
            print(f"Case {i}: expected {expected!r}, got {got!r}")
    print(f"{passed}/{len(cases)} checks passed")
    return passed == len(cases)

## Problem 11: Chunk Documents with Overlap

**The situation**

Your team is building a RAG assistant over long policy documents. The retriever works on small pieces of text, so every document is split into chunks first. If a sentence is cut at a chunk edge, the answer can lose context, so neighbouring chunks share a few words.

**Your task**

Write `chunk_document(text, chunk_size, overlap)`. It splits the text into chunks of up to `chunk_size` words, and each chunk repeats the last `overlap` words of the one before it.

**Example**

`chunk_document("a b c d e", 3, 1)` -> `["a b c", "c d e"]`

**Ask before coding**

- Do we split by words, characters, or model tokens?
- What should happen if overlap is equal to or bigger than the chunk size?
- Should a last chunk that adds no new words be dropped?

**Plan to say aloud**

1. Reject bad settings: chunk_size must be positive, overlap must be at least 0 and smaller than chunk_size.
2. Split the text into words.
3. Move a window forward by chunk_size minus overlap.
4. Join each window back into one string.
5. Stop as soon as a window reaches the last word.

### Prompt for your AI assistant

```text
Write a Python function `chunk_document(text: str, chunk_size: int, overlap: int) -> list[str]`. Split the text on whitespace into words. Each chunk has up to `chunk_size` words joined by single spaces. The next chunk starts `chunk_size - overlap` words later. Raise ValueError if chunk_size <= 0, or if overlap < 0 or overlap >= chunk_size. Stop after the chunk that contains the last word, so there is no tiny trailing chunk that only repeats words. Empty text returns an empty list. Standard library only. Show only the function.
```

**Check the AI's code for**

- Adds a tiny last chunk (for example `"e"`) that only repeats words already in the previous chunk. This is the most common slip, and it breaks the example above.
- Loops forever, or crashes with a range step of 0, when overlap equals chunk_size, because the check is missing.
- Splits on `" "` instead of `split()`, so double spaces and newlines create empty words.
- Chunks by characters even though the rule says words.

In [ ]:
def chunk_document(text: str, chunk_size: int, overlap: int) -> list[str]:
    """Split text into word chunks that overlap.

    Rules:
    - Raise ValueError if chunk_size <= 0, or overlap < 0, or overlap >= chunk_size.
    - Words come from text.split(). Each chunk is up to chunk_size words joined by one space.
    - The next chunk starts chunk_size - overlap words later.
    - Stop after the chunk that contains the last word (no tiny repeated tail).
    - Empty text returns [].
    """
    raise NotImplementedError

In [ ]:
cases = [
    (("a b c d e", 3, 1), ["a b c", "c d e"]),
    (("a b c d e f", 3, 1), ["a b c", "c d e", "e f"]),
    (("a b c d", 2, 0), ["a b", "c d"]),
    (("hello world", 5, 2), ["hello world"]),
    (("  a\n b   c ", 2, 1), ["a b", "b c"]),
    (("", 3, 1), []),
]
if run_checks(chunk_document, cases):
    try:
        chunk_document("a b", 2, 2)
        print("Bad overlap was accepted. It should raise ValueError.")
    except ValueError:
        print("Bad overlap rejected: ok")
    except Exception as e:
        print(f"Bad overlap raised {type(e).__name__}, expected ValueError")

### Reference solution

Try first. Then compare.

In [ ]:
def chunk_document_reference(text: str, chunk_size: int, overlap: int) -> list[str]:
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be at least 0 and smaller than chunk_size")
    words = text.split()
    step = chunk_size - overlap
    chunks = []
    for start in range(0, len(words), step):
        chunks.append(" ".join(words[start:start + chunk_size]))
        if start + chunk_size >= len(words):
            break  # this chunk already reached the last word
    return chunks


assert run_checks(chunk_document_reference, cases)
try:
    chunk_document_reference("a b", 2, 2)
    raise AssertionError("bad overlap should raise ValueError")
except ValueError:
    print("Bad overlap rejected: ok")

**Explain it in 30 seconds**

> "I split the text into words and slide a window across them. The window is chunk_size words wide and moves forward by chunk_size minus overlap, so neighbours share a few words. I reject settings that would make the window stand still, and I stop once a chunk reaches the last word so there is no useless tail. In production I would count model tokens instead of words."

**Edge cases to mention**

- Empty or whitespace-only text returns an empty list.
- Text shorter than one chunk returns a single chunk.
- overlap >= chunk_size would never move forward, so it raises instead.
- Tables and code blocks can be cut in awkward places by plain word splitting.

**If they ask "how would this work in production?"**

- Count tokens with the model's tokenizer, because the model's limit is in tokens, not words.
- Keep metadata on every chunk (document id, page, section, access roles) so later steps can cite and filter.
- Try a few chunk sizes against a small question set and keep the one that retrieves best. More overlap means more storage and embedding cost.

## Problem 19: Partial Document Ingestion

**The situation**

A 40-page PDF goes through OCR before it is indexed for search. Most pages come back with text, but a few fail. Throwing away the whole file wastes the good pages, and indexing it quietly hides the gaps.

**Your task**

Write `split_ingested_pages(results)`. It returns a dict of good pages (page number to text) and a sorted list of page numbers that failed, so good pages get indexed and failed ones get retried.

**Example**

`[{"page": 1, "text": "ok"}, {"page": 2, "error": "ocr_failed"}]` -> `({1: "ok"}, [2])`

**Ask before coding**

- Is it fine to index a document with some pages missing?
- Does a page with only whitespace count as a failure?
- What should happen to a result that has no page number?

**Plan to say aloud**

1. Start with an empty dict for good pages and an empty list for failures.
2. For each result, read page and text with `.get()`.
3. If the page number is not an int, skip it. We cannot retry a page we cannot name.
4. If the text is a non-empty string, keep it. Otherwise record the page as failed.
5. Return both, with failures sorted.

### Prompt for your AI assistant

```text
Write a Python function `split_ingested_pages(results: list[dict]) -> tuple[dict[int, str], list[int]]`. Each result may have the keys "page" (int), "text" (str) and "error". A page succeeds if "page" is an int and "text" is a string with some non-whitespace content. A page fails if "page" is an int and the text is missing, empty, whitespace-only, or not a string. Skip results without an int page number. Return (pages, failed) with failed sorted ascending. Use `.get()` so missing keys never raise. Show only the function.
```

**Check the AI's code for**

- Treats `"   "` as success because it only checks `text is not None`.
- Uses `result["text"]` and crashes with KeyError on the failed pages.
- Decides success by the absence of an `error` key, so a page with no error and no text counts as good.
- Puts results with no page number into the failed list as `None`.

In [ ]:
def split_ingested_pages(results: list[dict]) -> tuple[dict[int, str], list[int]]:
    """Split parser results into good pages and failed page numbers.

    Rules:
    - Good: "page" is an int and "text" is a string with non-whitespace content.
    - Failed: "page" is an int but text is missing, blank, or not a string.
    - Skip results without an int page number.
    - Return (pages, failed) with failed sorted ascending.
    """
    raise NotImplementedError

In [ ]:
cases = [
    (([{"page": 1, "text": "ok"}, {"page": 2, "error": "ocr_failed"}],), ({1: "ok"}, [2])),
    (([{"page": 1, "text": "   "}],), ({}, [1])),
    (([{"page": 3, "text": "c"}, {"page": 1, "text": None}, {"page": 2}],), ({3: "c"}, [1, 2])),
    (([{"text": "orphan text"}],), ({}, [])),
    (([],), ({}, [])),
]
run_checks(split_ingested_pages, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def split_ingested_pages_reference(results: list[dict]) -> tuple[dict[int, str], list[int]]:
    pages: dict[int, str] = {}
    failed: list[int] = []
    for result in results:
        page = result.get("page")
        text = result.get("text")
        if not isinstance(page, int):
            continue  # nothing to retry without a page number
        if isinstance(text, str) and text.strip():
            pages[page] = text
        else:
            failed.append(page)
    return pages, sorted(failed)


assert run_checks(split_ingested_pages_reference, cases)

**Explain it in 30 seconds**

> "I walk the parser results once and put each page into one of two buckets. A page is good only if it has a real page number and some real text. Anything else with a page number goes on the failed list, so we can retry it and tell the user exactly which pages are missing. The good pages still get indexed, so one bad page does not block the whole document."

**Edge cases to mention**

- Whitespace-only text counts as a failure, not a success.
- A result with no page number is skipped because there is nothing to retry. In production I would log it.
- Duplicate page numbers are not handled specially here. Worth asking which result should win.
- If most pages fail, the whole document should probably be flagged rather than partly indexed.

**If they ask "how would this work in production?"**

- Store the failure reason for each page so OCR problems can be grouped and fixed.
- Retry failed pages a limited number of times, then mark them as permanently missing.
- Show coverage to users (for example, 37 of 40 pages indexed) so the assistant never claims it read the full file.

## Problem 20: PII Redaction Validation

**The situation**

Before text is turned into embeddings, a redaction step replaces emails and phone numbers with placeholders like `[EMAIL]`. Redaction tools miss things. A second, separate check should confirm nothing obvious slipped through before the text leaves your control.

**Your task**

Write `pii_redaction_is_valid(text)`. It returns True when no email address or phone-like number is left, and False when one is still there.

**Example**

`"Contact test@example.com"` -> `False`, and `"Contact [EMAIL]"` -> `True`

**Ask before coding**

- Which kinds of PII must we check: only email and phone, or card numbers and IDs too?
- If the check fails, do we block the chunk or send it for review?
- Do we need to know where the leak is, or only that there is one?

**Plan to say aloud**

1. Write one simple pattern for emails: some characters, an @, then a domain with a dot.
2. Write one pattern for phone numbers: a run of digits that may have spaces, dashes, dots or brackets inside.
3. Search the whole text for each pattern.
4. Return True only if neither pattern is found.

### Prompt for your AI assistant

```text
Write a Python function `pii_redaction_is_valid(text: str) -> bool` that returns True only if the text contains no email address and no phone-like number. Email means `name@domain.tld`. Phone means an optional leading +, then a run of at least 9 characters that starts and ends with a digit and contains only digits, spaces, dashes, dots or brackets. Placeholders like [EMAIL] and [PHONE] must pass. Use `re.search`, standard library only. Show only the function.
```

**Check the AI's code for**

- Flips the meaning and returns True when PII is found. Read the name: valid means clean.
- Uses `re.match`, which only looks at the start of the string, instead of `re.search`.
- Makes the phone pattern so loose that short numbers like `Order 1234` or `v2.1.0` fail the check.
- Hard-codes `.com` in the email pattern and misses `.co.uk` or `.io`.

In [ ]:
import re


def pii_redaction_is_valid(text: str) -> bool:
    """Return True if no email or phone-like number is left in the text.

    Rules:
    - Email: name@domain.tld
    - Phone: optional +, then 9+ characters that start and end with a digit and
      contain only digits, spaces, dashes, dots or brackets.
    - Placeholders like [EMAIL] and [PHONE] pass.
    """
    raise NotImplementedError

In [ ]:
cases = [
    (("Contact test@example.com",), False),
    (("Contact [EMAIL] or [PHONE]",), True),
    (("Call me at +1 (415) 555-0100",), False),
    (("Email jane.doe@corp.co.uk today",), False),
    (("Order 1234 shipped in v2.1.0",), True),
    (("",), True),
]
run_checks(pii_redaction_is_valid, cases)

### Reference solution

Try first. Then compare.

In [ ]:
import re

EMAIL_PATTERN = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
PHONE_PATTERN = r"\+?\d[\d\s().-]{7,}\d"


def pii_redaction_is_valid_reference(text: str) -> bool:
    has_email = re.search(EMAIL_PATTERN, text) is not None
    has_phone = re.search(PHONE_PATTERN, text) is not None
    return not has_email and not has_phone


assert run_checks(pii_redaction_is_valid_reference, cases)

**Explain it in 30 seconds**

> "This is a safety net that runs after redaction. I search for two simple patterns, emails and phone-like digit runs, and the text passes only if neither shows up. Placeholders like [EMAIL] pass because they have no @ and no digits. A regex is only a baseline, so I call it a guard, not a guarantee."

**Edge cases to mention**

- Dates like 2024-01-15 look like phone numbers and cause false alarms.
- Hidden emails like `jane at example dot com` slip past a regex.
- Long customer IDs or order numbers can trip the phone pattern.
- Empty text has nothing to leak, so it returns True.

**If they ask "how would this work in production?"**

- Use a dedicated PII scanner and keep this regex as a cheap last check.
- Return what was found and where, so a reviewer can fix it fast.
- Quarantine failing chunks according to the customer's policy instead of embedding them, and count failures so redaction quality is visible.

## Wrap-up

- Each function sits between a raw document and the search index, so its mistakes become bad answers later.
- None of them crash on messy input. They either reject bad settings clearly or report what failed.
- Each one is a simple, testable baseline, and each has a clear upgrade path (tokens, retries, a real PII scanner).